In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import os

In [ ]:
N_COLAB = 'google.colab' in str(dir())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = "/content/drive/MyDrive/YOUR_FOLDER/pw_analysis"
else:
    WORK_DIR = "."

DATA_DIR = f"{WORK_DIR}/data_raw/NM geofiles"
OUT_DIR = f"{WORK_DIR}/outputs"

In [ ]:
nm_counties = gpd.read_file(
    f"{DATA_DIR}/nm_counties.geojson"
)
nm_counties["county_key"] = nm_counties["NAME"].str.strip().str.lower()
print(f"✓ {len(nm_counties)} counties")

In [ ]:
print(f"Shape:    {nm_counties.shape}")
print(f"CRS:      {nm_counties.crs}")           # should be EPSG:4326
print(f"Counties: {nm_counties['NAME'].tolist()}")


ax = nm_counties.plot(figsize=(8,6), edgecolor="white",
                      color="#1F456E", linewidth=0.5)
ax.set_title("NM county boundaries")
ax.axis("off")

In [ ]:
nm_counties = gpd.read_file(f"{DATA_DIR}/nm_counties.geojson")
nm_counties["county_key"] = nm_counties["NAME"].str.strip().str.lower()

# PW cross-validation data (pw volume + cluster + swd + oil counts)
validation_df = pd.read_csv(f"{WORK_DIR}/data_clean/data_processed/nm_county_summary.csv")
validation_df["county_key"] = validation_df["county_key"].str.strip().str.lower()

# Merge — [ county_key ]
nm_pw_geo = nm_counties.merge(validation_df, on="county_key", how="left")
nm_pw_geo["total_pw_vol"]     = nm_pw_geo["total_pw_vol"].fillna(0)
nm_pw_geo["dominant_cluster"] = nm_pw_geo["dominant_cluster"].fillna(-1)

# Confirm
print(f"Rows:            {len(nm_pw_geo)}")           # should be 33
print(f"Counties with PW data: {(nm_pw_geo['total_pw_vol'] > 0).sum()}")  # should be 5
print(nm_pw_geo[["NAME","total_pw_vol","dominant_cluster","swd_well_count"]].sort_values("total_pw_vol", ascending=False).head(10))

In [ ]:
swd_wells = gpd.read_file(f"{WORK_DIR}/data_clean/data_processed/ocd/nm_ocd_disposal_wells.geojson")
oil_wells = gpd.read_file(f"{WORK_DIR}/data_clean/data_processed/ocd/nm_ocd_oil_wells.geojson")
oil_sample = oil_wells.sample(n=min(5000, len(oil_wells)), random_state=42)

print(f"Counties:   {len(nm_pw_geo)}")
print(f"SWD wells:  {len(swd_wells):,}")
print(f"Oil sample: {len(oil_sample):,}")

In [ ]:
m = folium.Map(location=[34.5,-106.0], zoom_start=7,
               tiles="CartoDB positron")

# Layer 1: PW volume choropleth
folium.Choropleth(
    geo_data     = nm_pw_geo.to_json(),
    data         = nm_pw_geo,
    columns      = ["county_key","total_pw_vol"],
    key_on       = "feature.properties.county_key",
    fill_color   = "Blues",
    fill_opacity = 0.65,
    line_opacity = 0.3,
    nan_fill_color   = "#F1EFE8",
    nan_fill_opacity = 0.3,
    legend_name  = "PW volume — 5yr (bbl)",
    name          = "PW volume (choropleth)",
    show         = True
).add_to(m)

# Layer 2: cluster colour overlay
CLUSTER_COLORS = {-1:"#F1EFE8", 0:"#93C5FD", 1:"#34D399",
                   2:"#FBBF24", 3:"#EF4444"}

folium.GeoJson(
    nm_pw_geo,
    name = "PW clusters (colour by cluster)",
    style_function = lambda f: {
        "fillColor":   CLUSTER_COLORS.get(
            int(f["properties"].get("dominant_cluster",-1)), "#F1EFE8"),
        "color": "white", "weight": 0.8, "fillOpacity": 0.65
    },
    show = True,
    tooltip = folium.GeoJsonTooltip(
        fields  = ["NAME","dominant_cluster","total_pw_vol","swd_well_count"],
        aliases = ["County:","Cluster:","PW vol (5yr bbl):","SWD wells:"],
        localize = True
    )
).add_to(m)

# Hover tooltip on base layer
folium.GeoJson(
    nm_pw_geo,
    name = "County labels",
    style_function = lambda x: {"fillOpacity":0,"weight":0},
    tooltip = folium.GeoJsonTooltip(
        fields  = ["NAME","total_pw_vol","dominant_cluster","swd_well_count"],
        aliases = ["County:","PW vol (5yr bbl):","Cluster:","SWD wells:"],
        localize = True
    ), show = True
).add_to(m)
print("✓ County layers added")

In [ ]:
from folium.plugins import MarkerCluster

# Layer 3: SWD wells — teal dots
swd_group = folium.FeatureGroup(name="SWD wells", show=True)
for _, row in swd_wells.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#059669", fill=True,
        fill_color="#1D9E75", fill_opacity=0.8, weight=0.5,
        popup=folium.Popup(
            f"""{row.get('name','SWD Well')}
County: {str(row.get('county','')).title()}
Status: {row.get('status','')}""",
            max_width=200)
    ).add_to(swd_group)
swd_group.add_to(m)

# Layer 4: oil wells — grey, clustered for performance
oil_cluster = MarkerCluster(
    name="Oil wells (5k sample — cross-validation)", show=False) # Changed show=False to show=True
for _, row in oil_sample.iterrows():
    folium.CircleMarker(location=[row.geometry.y, row.geometry.x],
        radius=2, color="#64748B", fill=True,
        fill_color="#888780", fill_opacity=0.5, weight=0,
        popup=folium.Popup(
            f"""{row.get('name','Oil well')}
County: {str(row.get('county','')).title()}""",
            max_width=150)
    ).add_to(oil_cluster)
oil_cluster.add_to(m)

print(f"✓ SWD layer: {len(swd_wells):,} wells")
print(f"✓ Oil layer: {len(oil_sample):,} sampled wells")

In [ ]:
legend_html = """
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=True).add_to(m)
# Save
m.save(f"{OUT_DIR}/nm_combined_map3.html")
print("✓ Saved: nm_combined_map.html")

# Download to local machine
from google.colab import files
files.download(f"{OUT_DIR}/nm_combined_map3.html")